# ⚡ FreightQuote AI — Milestone 2
### Enterprise Multi-Agent Logistics Intelligence Platform
This notebook initializes environment dependencies, runs database schema migrations, validates clean architecture modules, and launches the Streamlit web application with an Ngrok public tunnel.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q streamlit streamlit-option-menu pyjwt bcrypt plotly pandas numpy scikit-learn joblib transformers accelerate bitsandbytes faker kaggle pyngrok python-dotenv

## Step 2 — Configure Secrets & Environment

In [ ]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
EMAIL_ADDRESS   = _get_secret("EMAIL_ADDRESS")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")

print("🔑 NGROK_AUTHTOKEN Configured:", "✅ Yes" if NGROK_AUTHTOKEN else "⚠️ No")
print("🔑 HF_TOKEN Configured:       ", "✅ Yes" if HF_TOKEN else "⚠️ No")
print("🔑 EMAIL_ADDRESS Configured:  ", "✅ Yes" if EMAIL_ADDRESS else "⚠️ Sandbox Mode")


## Step 3 — Database Initialization & Migration Verification

In [ ]:
import db, auth, config

# Run schema migration and initial user seeding
db.init_db()
db.seed_initial_users(auth.hash_txt, auth.check_txt, config.ADMIN_EMAIL, config.ADMIN_PASSWORD)

# Verify extended user directory columns
users = db.get_all_users()
print(f"✅ Database Initialized. Total Users: {len(users)}")
for u in users:
    print(f" - User: {u['email']} | Status: {u['account_status']} | Failed Attempts: {u['failed_attempts']}")


## Step 4 — Verify Clean Architecture Modules

In [ ]:
import train_ml_freight
import llm_engine_freight

# Instantiate ML Predictor and LLM Engine
predictor = train_ml_freight.FreightMLPredictor()
llm_engine = llm_engine_freight.LogisticsLLMEngine()

# Run test queries
est_quote = predictor.predict_freight_quote({"distance_nautical_miles": 4500, "cargo_weight_tons": 15, "container_type_code": 1})
res_llm = llm_engine.query_logistics_agent("What are the rates for Nhava Sheva to Rotterdam?")

print(f"✅ ML Module Verification: Estimated Quote = ${est_quote}")
print(f"✅ LLM Module Verification Response Snippet:\n{res_llm[:150]}...")


## Step 5 — Launch Streamlit Server with Ngrok Proxy

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Terminate any existing ngrok tunnels
ngrok.kill()

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    public_url = ngrok.connect(8501).public_url
    print(f"🚀 PUBLIC NGROK URL: {public_url}")
else:
    print("⚠️ NGROK_AUTHTOKEN not set. Running on local port 8501.")

# Start Streamlit background process
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
print("✅ Streamlit Server launched successfully!")
